# Libraries

In [1]:
# 1. Base Framework
!pip install -q -U \
    torch==2.10.0 \
    torchvision==0.25.0 \
    torchaudio==2.10.0 \
    --index-url https://download.pytorch.org/whl/cu128

# 2. torchao - force reinstall to actually replace Kaggle's broken copy
!pip install -q --no-deps --force-reinstall torchao \
    --index-url https://download.pytorch.org/whl/cu128

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.7 MB/s eta 0:00:00a 0:00:01


In [2]:
!pip install -q llmcompressor

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 7.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.6/563.6 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.6 MB/s eta 0:00:00


In [3]:
!pip install -q "transformers==4.57.6"

In [4]:
import random
import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
)

from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier

from huggingface_hub import (
    login,
    create_repo,
    upload_folder,
)

from kaggle_secrets import UserSecretsClient
from datasets import load_dataset

2026-05-01 13:23:34.394312: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777641814.786256      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777641814.890660      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777641815.843216      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777641815.843264      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777641815.843267      57 computation_placer.cc:177] computation placer alr

# Global Variables

In [ ]:
BASE_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_ID = "EdgeCompress01/Qwen3-4B-Instruct-2507-SparseGPT"
OUTPUT_DIR = "/kaggle/working/Qwen3-4B-Instruct-2507-SparseGPT-AWQ-4bit"
REPO_ID = f"EdgeCompress01/Qwen3-4B-Instruct-2507-SparseGPT-AWQ-4bit"
SEED = 42

# Functions

In [6]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

def format_chat(batch):
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        for messages in batch["messages"]
    ]
    return {"text": texts}

# Set Seed

In [7]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface Login

In [8]:
# --- 3. HUGGING FACE AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in")

Logged in


# Load Model & Tokenizer

In [ ]:
#Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    subfolder="Models/25",
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

#Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_ID, trust_remote_code=True)

config.json: 0.00B [00:00, ?B/s]

Models/50/model.safetensors:   0%|          | 0.00/8.04G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

# AWQ Quantization

**Configurations**

In [10]:
recipe = [
    AWQModifier(
        ignore=["lm_head", "embed_tokens"], 
        scheme="W4A16",
        targets=["Linear"]
    )
]

**Calibration Dataset**

In [11]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft").shuffle(seed=SEED).select(range(64))

formatted_dataset = dataset.map(
    format_chat,
    batched=True,
    remove_columns=dataset.column_names  # keep only "text"
)

README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00001-of-00003-0a1804bcb6(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00002-of-00003-ee46ed25cf(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/test_sft-00000-of-00001-f7dfac4afe5(…):   0%|          | 0.00/81.2M [00:00<?, ?B/s]

...

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

**Apply quantization**

In [12]:
oneshot(
    model=model,
    tokenizer=tokenizer,
    dataset=formatted_dataset,
    text_column="text",
    recipe=recipe,
    max_seq_length=512
)

2026-05-01T13:26:01.401981+0000 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.


Tokenizing:   0%|          | 0/64 [00:00<?, ? examples/s]

2026-05-01T13:26:01.880080+0000 | _make_sampler | WARNING - Requested 512 samples but the provided dataset only has 64 samples.
2026-05-01T13:26:01.881463+0000 | reset | INFO - Compression lifecycle reset
2026-05-01T13:26:01.884783+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-05-01T13:26:02.003511+0000 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...
2026-05-01T13:26:02.019523+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.0.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-05-01T13:26:02.020652+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.1.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-05-01T13:26:02.021582+0000 | _set_reso

W0501 13:26:02.165000 57 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
Smoothing:  33%|███▎      | 1/3 [00:17<00:35, 17.57s/it]                                                             
Grid search for model.layers.0.post_attention_layernorm:   0%|          | 0/20 [00:00<?, ?it/s]
Grid search for model.layers.0.post_attention_layernorm:   0%|          | 0/20 [00:01<?, ?it/s, best_error=8.154e-05]
Grid search for model.layers.0.post_attention_layernorm:   5%|▌         | 1/20 [00:01<00:34,  1.81s/it, best_error=8.154e-05]
Grid search for model.layers.0.post_attention_layernorm:   5%|▌         | 1/20 [00:03<00:34,  1.81s/it, best_error=5.770e-05]
Grid search for model.layers.0.post_attention_layernorm:  10%|█         | 2/20 [00:03<00:32,  1.81s/it, best_error=5.770e-05]
Grid s

2026-05-01T14:17:44.053330+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-05-01T14:17:44.054091+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

**Saving**

In [13]:
model.config._name_or_path = ""

model.save_pretrained(OUTPUT_DIR,skip_sparsity_compression_stats=False)#VERY IMPORTANT
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved successfully")

Calculating model sparsity: 100%|██████████| 903/903 [00:21<00:00, 42.29it/s]
Checking whether model follows 2:4 sparsity structure: 100%|██████████| 253/253 [00:22<00:00, 11.09it/s]
Compressing model: 252it [00:36,  6.92it/s]
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3970: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully


# PUSH TO HUGGING FACE

In [ ]:
create_repo(REPO_ID, repo_type="model", exist_ok=True)

upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    path_in_repo="Sparsity/25"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/EdgeCompress01/Qwen3-4B-Instruct-2507-WANDA-AWQ-4bit/commit/599a1bcb2f246a07a6e2d256d80761916d6a85fa', commit_message='Upload folder using huggingface_hub', commit_description='', oid='599a1bcb2f246a07a6e2d256d80761916d6a85fa', pr_url=None, repo_url=RepoUrl('https://huggingface.co/EdgeCompress01/Qwen3-4B-Instruct-2507-WANDA-AWQ-4bit', endpoint='https://huggingface.co', repo_type='model', repo_id='EdgeCompress01/Qwen3-4B-Instruct-2507-WANDA-AWQ-4bit'), pr_revision=None, pr_num=None)